# Verify VAE-GAN

Inspect the exported latent-mean encoder, reconstructions, prior samples and interpolation. Image quality alone does not establish usefulness for robot control.


In [ ]:
from pathlib import Path
import sys
import torch
import matplotlib.pyplot as plt
from IPython.display import clear_output, display

# Run from this notebook's sensorprocessing directory.
if str(Path("..").resolve()) not in sys.path:
    sys.path.insert(0, str(Path("..").resolve()))
from exp_run_config import Config
Config.PROJECTNAME = "BerryPicker"
from sensorprocessing.conv_vae_neo import ConvVAENeo, make_dataloaders
from sensorprocessing.vae_gan_training import train
from sensorprocessing.vae_gan_visualization import plot_history, plot_reconstructions, plot_images
from sensorprocessing.sp_factory import create_sp
from training_harness.checkpoints import model_file

experiment = "sensorprocessing_vae_gan"
run = "sp_vae_gan_128_256px"
expruns_path = None  # Optional existing external configuration directory
results_path = None  # Optional existing results directory
if expruns_path is not None:
    if not Path(expruns_path).is_dir():
        raise FileNotFoundError(expruns_path)
    Config().set_exprun_path(expruns_path)
    for group in [experiment, "sensorprocessing_conv_vae_neo", "demonstration", "robot_al5d"]:
        Config().copy_experiment(group)
if results_path is not None:
    if not Path(results_path).is_dir():
        raise FileNotFoundError(results_path)
    Config().set_results_path(results_path)
exp = Config().get_experiment(experiment, run, creation_style="exist-ok")
device = Config().runtime["device"]


In [ ]:
# Load the best exported VAE, not the last training state.
model = ConvVAENeo(exp).to(device)
model.load_state_dict(torch.load(model_file(exp), map_location=device, weights_only=True))
model.eval()
print(f"Loaded best VAE: {model_file(exp)}")

# Alternatively load the VAE portion of a retained intermediate full checkpoint:
# checkpoint_path = Path(exp["data_dir"]) / "checkpoints" / "epoch_000010.pt"
# checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=True)
# model.load_state_dict({key.removeprefix("vae."): value
#                        for key, value in checkpoint["model_state_dict"].items()
#                        if key.startswith("vae.")})
# model.eval()


In [ ]:
loaders = make_dataloaders(exp)
validation_dataset = loaders[1].dataset
images = torch.stack([validation_dataset[i] for i in range(min(4, len(validation_dataset)))]).to(device)
# Local CPU generator: these samples remain fixed without altering training RNG.
latent = torch.randn(len(images), exp["latent_size"], generator=torch.Generator().manual_seed(123)).to(device)


In [ ]:
display(plot_reconstructions(model, images, latent))


In [ ]:
with torch.no_grad():
    endpoints = model.encode(images)
    if len(endpoints) < 2:
        raise ValueError("Interpolation needs at least two validation images")
    alpha = torch.linspace(0, 1, 8, device=device).unsqueeze(1)
    interpolated = model.decode((1 - alpha) * endpoints[:1] + alpha * endpoints[1:2])
display(plot_images([interpolated], ["Latent interpolation"]))


In [ ]:
processor = create_sp(exp)
encoding = processor.process(images[:1])
with torch.no_grad():
    expected = model.encode(images[:1]).squeeze(0).cpu().numpy()
import numpy as np
np.testing.assert_allclose(encoding, expected, rtol=1e-5, atol=1e-6)
print("Runtime latent shape:", encoding.shape)
